# 🚗 Vehicle Fuel Efficiency Prediction
## Notebook 3: Data Cleaning

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

df = pd.read_csv('../data/auto-mpg.csv', na_values='?')
print(f'Loaded: {df.shape} | Missing in horsepower: {df["horsepower"].isna().sum()}')

## 3.1 Handle Missing Values

In [ ]:
# --- Inspect the rows with missing horsepower ---
print('Rows with missing horsepower:')
display(df[df['horsepower'].isna()])

# Strategy: Impute with MEDIAN (robust to outliers)
hp_median = df['horsepower'].median()
df['horsepower'].fillna(hp_median, inplace=True)

print(f'\nImputed missing horsepower with median: {hp_median}')
print(f'Missing values remaining: {df.isnull().sum().sum()}')

## 3.2 Correct Data Types

In [ ]:
# horsepower should be float (was object due to '?')
df['horsepower'] = df['horsepower'].astype(float)

# origin: categorical (1=USA, 2=Europe, 3=Japan)
df['origin'] = df['origin'].astype('category')

# model_year: keep as int (will engineer later)
print('Updated dtypes:')
print(df.dtypes)

## 3.3 Detect & Handle Outliers

In [ ]:
numeric_cols = ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].boxplot(df[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor='#4C72B0', alpha=0.7),
                    medianprops=dict(color='red', linewidth=2),
                    flierprops=dict(marker='o', color='orange', markersize=6))
    axes[i].set_title(f'{col}', fontsize=12, fontweight='bold')
    axes[i].set_xticks([])

plt.suptitle('Outlier Detection — Box Plots', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../plots/02_outlier_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# IQR-based outlier detection (log, don't cap — small dataset)
outlier_report = {}
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_report[col] = {'lower_bound': round(lower, 2),
                            'upper_bound': round(upper, 2),
                            'n_outliers': n_outliers}

outlier_df = pd.DataFrame(outlier_report).T
print('IQR Outlier Summary:')
display(outlier_df)

# Decision: Dataset is small (398 rows) — retain outliers.
# Tree-based models are inherently robust to outliers.
print('\n✅ Decision: Retaining outliers (tree models handle them natively)')

## 3.4 Remove Redundant Features

In [ ]:
# 'car_name' is a unique identifier — not predictive
# Extract brand as a potential feature first
df['brand'] = df['car_name'].apply(lambda x: str(x).strip().split()[0].lower())

# Standardize known brand aliases
brand_map = {
    'chevroelt': 'chevrolet',
    'chevy': 'chevrolet',
    'toyouta': 'toyota',
    'maxda': 'mazda',
    'vokswagen': 'volkswagen',
    'vw': 'volkswagen',
}
df['brand'] = df['brand'].replace(brand_map)

print(f'Top 15 brands:')
print(df['brand'].value_counts().head(15))

# Drop car_name (keep brand)
df.drop(columns=['car_name'], inplace=True)
print('\nDropped car_name, kept brand ✓')

In [ ]:
# Save cleaned dataset
df.to_csv('../data/auto-mpg-cleaned.csv', index=False)
print(f'Cleaned dataset saved: {df.shape}')
df.head()